In [15]:
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

# Student Identity Inputs
LAST_NAME = "AMISTAD"
STUDENT_ID = "TUPM-26-1938"
SEED_DIGIT = int(STUDENT_ID[-1])  # 8

# Data Generation Rules
TARGET_YEAR = 2015 + SEED_DIGIT  # 2023

# File Paths
DATASET_PATH = Path("Global_Cybersecurity_Threats_2015-2024.csv")
FILTERED_CSV_PATH = Path("filtered_cybersecurity_2023.csv")
SUMMARY_JSON_PATH = Path("cybersecurity_summary_2023.json")
BACKUP_JSON_PATH = Path("cybersecurity_summary_2023_backup.json")
REPORT_TXT_PATH = Path("cybersecurity_report_2023.txt")

# Data Containers
total_dataset_records = 0
target_year_incidents = []
attack_type_counts = Counter()
total_financial_loss = 0.0

# 1. Read CSV line-by-line and parse records
with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_dataset_records += 1

        try:
            raw_timestamp = row.get("Timestamp", row.get("Year", "")).strip()
            year_str = raw_timestamp.split("-")[0].strip()

            if not year_str.isdigit():
                continue

            year_val = int(year_str)

            if year_val == TARGET_YEAR:
                attack_type = row.get("Attack Type", "").strip()
                loss_str = row.get("Financial Loss (in Million $)", "").strip()

                if not loss_str:
                    continue

                loss_val = float(loss_str)

                record = {
                    "Timestamp": raw_timestamp,
                    "Attack Type": attack_type,
                    "Financial Loss (in Million $)": loss_val,
                }

                target_year_incidents.append(record)
                attack_type_counts[attack_type] += 1
                total_financial_loss += loss_val

        except (ValueError, TypeError):
            continue

# Compute summary statistics
incidents_count = len(target_year_incidents)
avg_financial_loss = (total_financial_loss / incidents_count) if incidents_count > 0 else 0.0

top_5_costliest = sorted(
    target_year_incidents,
    key=lambda x: x["Financial Loss (in Million $)"],
    reverse=True
)[:5]

# Export filtered records to CSV
fieldnames = ["Timestamp", "Attack Type", "Financial Loss (in Million $)"]
with open(FILTERED_CSV_PATH, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(target_year_incidents)

# Export JSON summary & create verified backup
summary_data = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "target_year": TARGET_YEAR,
    "total_records_in_dataset": total_dataset_records,
    "incidents_in_target_year": incidents_count,
    "count_per_attack_type": dict(attack_type_counts),
    "total_financial_loss_usd_millions": round(total_financial_loss, 2),
    "average_financial_loss_usd_millions": round(avg_financial_loss, 2),
}

with open(SUMMARY_JSON_PATH, mode="w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

shutil.copy(SUMMARY_JSON_PATH, BACKUP_JSON_PATH)
backup_verified = BACKUP_JSON_PATH.exists()

print("=" * 70)
print(f"{'EXERCISE 3: CYBERSECURITY INCIDENT REPORT':^70}")
print("=" * 70)
print(f"Target Year            : {TARGET_YEAR}")
print(f"Total Dataset Records  : {total_dataset_records}")
print(f"Incidents in {TARGET_YEAR}     : {incidents_count}")
print(f"Attack Type Counts     : {dict(attack_type_counts)}")
print(f"Total Financial Loss   : ${total_financial_loss:,.2f} Million")
print(f"Avg Financial Loss     : ${avg_financial_loss:,.2f} Million")
print("-" * 70)
print("Five Costliest Incidents:")
for idx, inc in enumerate(top_5_costliest, start=1):
    inc_id = inc.get("Incident ID", "N/A")
    loss = inc.get("Financial Loss (in Million $)", 0.0)
    atk = inc.get("Attack Type", "Unknown")
    print(f"  {idx}. Loss: ${loss:>6.2f}M | Type: {atk}")
print("-" * 70)
print(f"Filtered CSV File      : {FILTERED_CSV_PATH.name}")
print(f"JSON Summary File      : {SUMMARY_JSON_PATH.name}")
print(f"Backup Verification    : {'VERIFIED' if backup_verified else 'FAILED'}")
print(f"Text Report File       : {REPORT_TXT_PATH.name}")
print("=" * 70)

              EXERCISE 3: CYBERSECURITY INCIDENT REPORT               
Target Year            : 2023
Total Dataset Records  : 3000
Incidents in 2023     : 315
Attack Type Counts     : {'DDoS': 51, 'Ransomware': 50, 'Phishing': 46, 'Malware': 55, 'Man-in-the-Middle': 50, 'SQL Injection': 63}
Total Financial Loss   : $15,958.08 Million
Avg Financial Loss     : $50.66 Million
----------------------------------------------------------------------
Five Costliest Incidents:
  1. Loss: $ 99.72M | Type: Malware
  2. Loss: $ 99.71M | Type: Man-in-the-Middle
  3. Loss: $ 99.64M | Type: DDoS
  4. Loss: $ 99.62M | Type: DDoS
  5. Loss: $ 99.19M | Type: SQL Injection
----------------------------------------------------------------------
Filtered CSV File      : filtered_cybersecurity_2023.csv
JSON Summary File      : cybersecurity_summary_2023.json
Backup Verification    : VERIFIED
Text Report File       : cybersecurity_report_2023.txt
